In [1]:
import subprocess
import sys

# Function to install boto3
def install_boto3():
    try:
        # Check if boto3 is already installed
        import boto3
        print("boto3 is already installed")
    except ImportError:
        # If boto3 is not installed, install it using pip
        print("Installing boto3...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3"])

# Call the function to install boto3
install_boto3()

boto3 is already installed


In [2]:
import os
import boto3
from botocore.exceptions import NoCredentialsError

# S3 details
s3_bucket = 'emrdemomuf'
s3_key = 'code/mysql-connector-java-5.1.40-bin.jar'
local_path = '/tmp/mysql-connector-java-5.1.40-bin.jar'

# Function to download from S3 if file doesn't exist locally
def download_from_s3_if_needed():
    if not os.path.exists(local_path):
        print(f"File {local_path} not found locally. Downloading from S3...")
        
        # Initialize S3 client
        s3 = boto3.client('s3')

        try:
            # Download file from S3
            s3.download_file(s3_bucket, s3_key, local_path)
            print(f"Downloaded {s3_key} from S3 to {local_path}")
        except NoCredentialsError:
            print("Credentials not available for S3.")
        except Exception as e:
            print(f"Error downloading file: {str(e)}")
    else:
        print(f"File {local_path} already exists locally.")

# Call the function
download_from_s3_if_needed()


File /tmp/mysql-connector-java-5.1.40-bin.jar already exists locally.


In [3]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("MySQL Query") \
    .config("spark.jars", "/tmp/mysql-connector-java-5.1.40-bin.jar") \
    .getOrCreate()

In [4]:
# MySQL database connection details
url = "jdbc:mysql://db.fusi24.com:3306/employees"

table = "employees"
properties = {
    "user": "muf", # Replace with your MySQL username
    "password": "Training123MUF", # Replace with your MySQL password
    "driver": "com.mysql.jdbc.Driver"
}

# Partitioning options
partition_column = "emp_no"     # Column used for partitioning
lower_bound = 1                 # Minimum value in the partition column
upper_bound = 1000000           # Maximum value in the partition column
num_partitions = 10             # Number of partitions to divide the data

# Read the employees table into a Spark DataFrame
# df = spark.read.jdbc(url=url, table=table, properties=properties)

df = spark.read.jdbc(
    url=url,
    table=table,
    properties=properties,
    column=partition_column,
    lowerBound=lower_bound,
    upperBound=upper_bound,
    numPartitions=num_partitions
)

# Show the first few rows of the employees table
df.count()

300029

In [5]:
# Query the DataFrame with PySpark
df.filter(df['first_name'] == 'Mayuko').show()

# Alternatively, register the DataFrame as a temporary SQL table and use Spark SQL
df.createOrReplaceTempView("employees_table")

# Query with Spark SQL
spark.sql("SELECT * FROM employees_table WHERE first_name = 'Mayuko'").show()

+------+----------+----------+--------------+------+----------+
|emp_no|birth_date|first_name|     last_name|gender| hire_date|
+------+----------+----------+--------------+------+----------+
| 10020|1952-12-24|    Mayuko|       Warwick|     M|1991-01-26|
| 11952|1956-04-11|    Mayuko|         Munro|     F|1995-12-19|
| 13443|1954-07-22|    Mayuko|         Puppo|     M|1985-03-06|
| 13780|1958-07-16|    Mayuko|        Bodoff|     M|1994-10-07|
| 13978|1959-09-18|    Mayuko|       Bardell|     F|1988-05-16|
| 14869|1956-10-21|    Mayuko|      Redmiles|     M|1991-08-19|
| 15400|1960-08-21|    Mayuko|       Berendt|     M|1989-05-04|
| 16096|1955-11-25|    Mayuko|      Wallrath|     M|1990-08-11|
| 16898|1960-02-08|    Mayuko|     Pocchiola|     F|1990-06-15|
| 17049|1956-11-04|    Mayuko|Georgakopoulos|     F|1992-12-17|
| 18252|1963-09-10|    Mayuko|    Besancenot|     M|1987-08-15|
| 20700|1964-04-14|    Mayuko|         Krohm|     M|1993-03-30|
| 21026|1953-08-03|    Mayuko|       Por

In [6]:
# Stop the Spark session when done
spark.stop()